# 01 · Practice-level prescribing from the NHSBSA English Prescribing Dataset
Downloads 2016 items, quantity and ADQ per London practice × BNF code for each condition through the NHSBSA open-data SQL API.
The logic is in `01_epd_download_workers.py`.

**Inputs:** `data/raw_data/bnf_lists/*.csv` (condition → BNF codes).
**Outputs:** `data/raw_data/prescriptions_epd/{condition}_{YYYYMM}.csv`, `practices_{YYYYMM}.csv`, `epd_2016_{condition}.csv`.
**Runtime:** about 40 minutes for a year; it resumes where it stopped (a month is done when its CSV exists). Run from this folder.
**Order:** step 1 downloads the practices in the NHS London region. Step 2 adds practices outside the region that hold at least 100 London patients; their list is written by `02_allocate_to_areas.ipynb` (step 1), so run 02 step 1 between the two.

## 1. Download the London-region practices → SAVE

In [ ]:
import importlib
import pandas as pd

w = importlib.import_module('01_epd_download_workers')
YEAR = 2016
w.run_year(YEAR)

## 2. Practices outside the region (needs `extra_practices_2016.txt` from 02) → SAVE

In [ ]:
extra_path = w.OUT_DIR / f'extra_practices_{YEAR}.txt'
if extra_path.exists():
    extra = extra_path.read_text().split()
    w.run_year(YEAR, practice_codes=extra, tag='__extra')
    print(len(extra), 'extra practices downloaded')

## 3. Merge the months → SAVE

In [ ]:
tags = ('', '__extra') if extra_path.exists() else ('',)
for cond in w.CONDITIONS:
    df = w.merge_year(YEAR, cond, tags)
    print(f'{cond:13s} rows {len(df):7d}  practices {df.PRACTICE_CODE.nunique():5d}  items {df["items"].sum():12,.0f}')

## Validation

In [ ]:
pr = pd.concat([pd.read_csv(f) for f in sorted(w.OUT_DIR.glob(f'practices_{YEAR}??.csv'))])
print('months:', pr.YEAR_MONTH.nunique(), '| practices per month:', pr.groupby('YEAR_MONTH').PRACTICE_CODE.nunique().describe()[['min', 'max']].to_dict())
pr.groupby('PCO_NAME')['items'].sum().sort_values(ascending=False).head(10)